# JN3 - Build the spine + units (the corrected signal)

**Curriculum notebook 3 of 6.** Now we turn permits into *buildings* and count their *new* units. This is where the most dangerous trap in civic data lives: **the same column means different things on different permit types.** Get it wrong and you invent thousands of homes that already existed - or you silently drop real ones.

> Clonable + **read-only** - demonstrates the real `housing_predicates.net_units`.

## (run first) Colab setup

Fetches the data + shared modules from R2. **No-op if you already have the repo locally** (it detects a checkout and skips). On Colab / a bare session it recreates the minimal repo layout under the working directory so the config cell below finds everything unchanged.

In [1]:
# === COLAB BOOTSTRAP - fetch curriculum data + modules from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import sys, urllib.request, urllib.parse, tarfile, subprocess

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
USE_CLEAN = False   # False: raw .xlsx path (JN1's messy-data lesson).  True (skip-ingest): permits_clean.*

_here = Path.cwd()
_have_repo = (_here/'scripts'/'build_v2').exists() or any((p/'scripts'/'build_v2').exists() for p in _here.parents)

def _get(url):
    # r2.dev sits behind Cloudflare, which 403s the default 'Python-urllib' User-Agent; send a browser UA.
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if _have_repo:
    print('local repo detected - no fetch needed')
else:
    try:
        import pyarrow  # the parquet / USE_CLEAN path needs it; Colab has pandas, maybe not pyarrow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
    def _fetch(url, dest):
        dest = Path(dest)
        if dest.exists():
            return                                   # cached: re-runs don't re-download
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(_get(url)); print('fetched', dest.name)
    # 1) shared modules -> ./scripts/...  (the config-cell repo-root walk then finds scripts/build_v2)
    if not (_here/'scripts'/'build_v2').exists():
        Path('modules.tgz').write_bytes(_get(f'{R2}/curriculum_modules.tar.gz'))
        _tar = tarfile.open('modules.tgz')
        try: _tar.extractall(_here, filter='data')      # py3.12+: safe extract, no deprecation warning
        except TypeError: _tar.extractall(_here)         # older python has no filter arg
        _tar.close(); Path('modules.tgz').unlink(missing_ok=True)   # tidy: drop the intermediate tarball
        print('extracted modules -> ./scripts/')
    # 2) data -> the SAME relative paths the notebooks use (raw .xlsx AND clean exports, both fetched)
    for rel in ['data/raw/cpra-downloads/BP_Annual Permit Report-2018-2022.xlsx',
                'data/raw/cpra-downloads/BP_Annual Permit Report-2023-2025.xlsx',
                'databases/hcd_apr_mirror_2026-06-17_fresh.db',
                'databases/hcd_apr_mirror.db',
                'data/processed/permits_clean.csv',
                'data/processed/permits_clean.parquet',
                'data/processed/permits_clean_README.md']:
        _fetch(f"{R2}/data/{urllib.parse.quote(rel.split('/')[-1])}", _here/rel)   # quote -> %20 for the spaced .xlsx names
    print('curriculum bundle ready (fetched from R2)')


local repo detected - no fetch needed


## Config

In [2]:
# === CONFIG - point this at YOUR city's permit data (this notebook is clonable) ===
from pathlib import Path
import sys, glob
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts' / 'build_v2').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
PERMIT_GLOB = str(REPO_ROOT / 'data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx')
HEADER_ROW  = 7
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'build_v2'))
print('repo root:', REPO_ROOT)


repo root: /Users/johngage/berkeley-data


## Recap: JN1's clean feed + JN2's housing filter

In [3]:
import pandas as pd
from collections import defaultdict
from housing_predicates import is_housing, net_units   # the REAL shared predicates (JN3 demonstrates them)
from s0_keys import normalize_address                   # the address key from JN2

def load(path):
    d = pd.read_excel(path, dtype=str, header=HEADER_ROW); d.columns = [str(c).strip() for c in d.columns]; return d
df = pd.concat([load(f) for f in glob.glob(PERMIT_GLOB)], ignore_index=True)
df = df[df['PermitNumber'].notna()].copy()
df['isnew'] = df['Work Type'].astype(str).str.strip() == 'New'
df['resi'] = [is_housing(o, u, n, a) for o, u, n, a in zip(df['OccType'], df['UnitsAdded'], df['NumberUnits'], df['ADU'])]
resi = df[df['resi']].copy()
print(f'{len(df):,} permits -> {len(resi):,} housing rows')


32,202 permits -> 31,314 housing rows


## The corrected unit signal - `net_units` (import it, don't reinvent)

`housing_predicates.net_units(is_new, units_added, number_units, adu_flag)` is the **one** rule the whole pipeline uses. Read its four branches - each exists because of a real bug it prevents:

| branch | rule | why |
|---|---|---|
| 1 | `UnitsAdded` if `>0` | the explicit *net add* - trust it when present |
| 2 | else `NumberUnits` if **New** | a new building: *all* its units are new |
| 3 | else `min(NumberUnits, 2)` if **ADU** | the ADU/JADU - but **capped at 2**, because a permit on an 82-unit building flagged ADU reports `NumberUnits=82` (the existing stock), not 82 new ADUs |
| 4 | **else `0`** | a plain alteration's `NumberUnits` is the **existing** count - **not new housing** |

In [4]:
print(net_units.__doc__)

CREATION + COUNT (narrow). Net-new dwelling units this permit creates (MAX across a building's
    permits = REV/phase guard):
      - UnitsAdded if present (explicit net add);
      - else NumberUnits if Work Type=New (a new building: all its units are new);
      - else min(NumberUnits, 2) if ADU=Yes (the ADU/JADU — capped, since a permit on an 82-unit
        building flagged ADU reports NumberUnits=82, the EXISTING stock, not 82 new ADUs);
      - else 0 (a plain alteration's NumberUnits is the EXISTING unit count, NOT new housing).
    is_new is a boolean (py/np); adu_flag is read through the shared is_adu (bool/'Yes' tolerant).


## THE CORE LESSON: same column, different meaning

`NumberUnits` is the trap. On a **New** permit it is *new units*. On an **Alteration** it is the **building that already exists**. A naive rule - "use `UnitsAdded`, else `NumberUnits`" - cannot tell them apart, so it adds existing stock as if it were new.

Watch it on a real permit: **B2023-02847**, an alteration at **2024 Durant** whose work was *"remove electric fireplace, replace light fixtures"* - in a building that **already had 99 units**.

In [5]:
r = df[df['PermitNumber'] == 'B2023-02847'].iloc[0]
ua, nu = float(r['UnitsAdded'] or 0), float(r['NumberUnits'] or 0)
naive   = ua if ua > 0 else nu                                  # 'UnitsAdded, else NumberUnits'
correct = net_units(r['isnew'], r['UnitsAdded'], r['NumberUnits'], r['ADU'])
print(f"B2023-02847  WorkType={r['Work Type']}  NumberUnits={r['NumberUnits']}  UnitsAdded={r['UnitsAdded']}")
print(f"  work: {str(r['WorkDescription'])[:70]}")
print(f'  NAIVE  ua-else-nu     -> {naive:.0f} new units   <-- 99 PHANTOM homes (the building already existed)')
print(f'  net_units (branch 4)  -> {correct:.0f} new units   <-- correct: an alteration adds no new housing')

B2023-02847  WorkType=Alteration  NumberUnits=99  UnitsAdded=0
  work: Remove electric fireplace, replace light fixtures and relocate various
  NAIVE  ua-else-nu     -> 99 new units   <-- 99 PHANTOM homes (the building already existed)
  net_units (branch 4)  -> 0 new units   <-- correct: an alteration adds no new housing


### The scale of the trap

Apply both rules across the whole feed. The naive rule reads existing stock as new on *thousands* of alteration permits. The pipeline groups permits to buildings with **MAX** (not SUM - a later lesson), which absorbs most of it, but phantom units still leak into the spine.

In [6]:
def fnum(x):
    try: v = float(str(x).replace(',', ''))
    except: return 0.0
    return v if v == v else 0.0          # NaN guard (a blank cell reads as 'nan')
def nz(v): return v if v == v else 0.0
naive_permit   = sum(fnum(r.UnitsAdded) if fnum(r.UnitsAdded) > 0 else fnum(r.NumberUnits) for r in resi.itertuples())
correct_permit = sum(nz(net_units(r.isnew, r.UnitsAdded, r.NumberUnits, r.ADU)) for r in resi.itertuples())
print(f'per-permit, summed across the feed:  naive {naive_permit:,.0f}  vs  net_units {correct_permit:,.0f}')
print(f'  -> {naive_permit - correct_permit:,.0f} phantom units the naive rule invents from existing stock')

per-permit, summed across the feed:  naive 90,446  vs  net_units 46,320
  -> 44,126 phantom units the naive rule invents from existing stock


## The other direction: too-narrow a read silently DROPS real housing

Branch 1 alone ("only `UnitsAdded`") looks safe - but ADUs are coded with their count in **`NumberUnits`** and `UnitsAdded=0`. A `UnitsAdded`-only rule reads them as **zero** and erases real homes. Branch 3 (`min(NumberUnits, 2)` when ADU) recovers them. Real example: **B2021-03756**, 1534 Oregon - `ADU=Yes`, `NumberUnits=2`, `UnitsAdded=0`.

In [7]:
r = df[df['PermitNumber'] == 'B2021-03756'].iloc[0]
ua_only = float(r['UnitsAdded'] or 0)
correct = net_units(r['isnew'], r['UnitsAdded'], r['NumberUnits'], r['ADU'])
print(f"B2021-03756  ADU={r['ADU']}  NumberUnits={r['NumberUnits']}  UnitsAdded={r['UnitsAdded']}")
print(f'  UnitsAdded-only -> {ua_only:.0f}  <-- a real ADU DROPPED')
print(f'  net_units (branch 3, min(nu,2)) -> {correct:.0f}  <-- recovered')
n_adu = sum(1 for r in resi.itertuples() if str(r.ADU).strip().lower() == 'yes'
            and fnum(r.NumberUnits) > 0 and fnum(r.UnitsAdded) == 0 and not r.isnew)
print(f'\nADU permits whose count lives in NumberUnits (recovered by branch 3): {n_adu}  (the ~265-ADU tail)')

B2021-03756  ADU=Yes  NumberUnits=2  UnitsAdded=0
  UnitsAdded-only -> 0  <-- a real ADU DROPPED
  net_units (branch 3, min(nu,2)) -> 2  <-- recovered

ADU permits whose count lives in NumberUnits (recovered by branch 3): 373  (the ~265-ADU tail)


## Build the spine: group permits to buildings, units = MAX over the building's permits

A building = all its housing permits under one **address key** (JN2). Its unit count is the **MAX** `net_units` over those permits - never the SUM, because phased permits (`Phase I`, `Phase II`) each repeat the *whole-building* count. A building is in the **spine** if it has any New permit or any net-new units.

In [8]:
spine = defaultdict(lambda: {'units': 0.0, 'hasnew': False})
for r in resi.itertuples(index=False):
    st = r.StreetType; st = '' if (st is None or str(st).strip().lower() == 'nan') else str(st)
    k = normalize_address(f'{r.StreetNumber} {r.StreetName} {st}'.strip())
    if not k.number: continue
    key = (k.number, k.street, k.stype)
    spine[key]['units'] = max(spine[key]['units'], net_units(r.isnew, r.UnitsAdded, r.NumberUnits, r.ADU))
    if r.isnew: spine[key]['hasnew'] = True
spine = {k: b for k, b in spine.items() if b['hasnew'] or b['units'] > 0}
print(f"spine buildings: {len(spine)}   (matches the pipeline's S1 spine of 1385)")
print(f"spine net-new units: {sum(b['units'] for b in spine.values()):,.0f}")

spine buildings: 1385   (matches the pipeline's S1 spine of 1385)
spine net-new units: 5,705


## (warning) Known limitation - foreshadowing JN6

This grouping keys a building by its **address**. A development with **two buildings at one address** (different parcels, different completion years) collapses into **one** building, and MAX keeps only the larger. Watch **2352 Shattuck** (Logan Park = a 135-unit North building **and** a 69-unit South building):

In [9]:
k = normalize_address('2352 Shattuck Ave')
print(f'2352 Shattuck in this spine: {spine[(k.number, k.street, k.stype)]}')
print('  -> ONE building, 135u (the North). The South 69u is hidden by MAX-on-one-address.')
print('  -> This is NOT fixed here. You will REDISCOVER it in JN6 when the scorecard residual')
print('     surfaces it -- a collapse made into a taught lesson, not a hidden bug.')

2352 Shattuck in this spine: {'units': 135.0, 'hasnew': True}
  -> ONE building, 135u (the North). The South 69u is hidden by MAX-on-one-address.
  -> This is NOT fixed here. You will REDISCOVER it in JN6 when the scorecard residual
     surfaces it -- a collapse made into a taught lesson, not a hidden bug.


## Checkpoint

In [10]:
# 1) the spine reproduces the pipeline's S1 building count
assert len(spine) == 1385, f'spine {len(spine)} != 1385'
# 2) the else-0 guard: a real alteration adds 0 new units (existing stock NOT counted)
alt = df[df['PermitNumber'] == 'B2023-02847'].iloc[0]
assert net_units(alt['isnew'], alt['UnitsAdded'], alt['NumberUnits'], alt['ADU']) == 0
# 3) the ADU tail is present (not dropped): a NumberUnits-coded ADU contributes its units
adu = df[df['PermitNumber'] == 'B2021-03756'].iloc[0]
assert net_units(adu['isnew'], adu['UnitsAdded'], adu['NumberUnits'], adu['ADU']) == 2
assert n_adu > 250   # the ADU tail survives

print('CHECKPOINT PASS')
print(f'  spine = {len(spine)} buildings (== S1)  -  alteration B2023-02847 -> 0 new (else-0 guard)')
print(f'  ADU tail present: {n_adu} NumberUnits-coded ADUs recovered (1534 Oregon -> 2 units)')

CHECKPOINT PASS
  spine = 1385 buildings (== S1)  -  alteration B2023-02847 -> 0 new (else-0 guard)
  ADU tail present: 373 NumberUnits-coded ADUs recovered (1534 Oregon -> 2 units)


**JN3 done.** You have a spine of buildings with a *corrected* unit count - phantom existing-stock kept out, real ADUs kept in. **Next - JN4:** dated milestone events and the completion stage (when is a building actually *done*?).